In [ ]:
PROGRAM hw12
!=====================
implicit none
character(50) :: filename
integer, parameter :: num_files = 200
integer, parameter :: num_levels = 37
real, dimension(num_levels) :: lev, q, u, v
integer :: i, j, k,  status, n, LREC
CHARACTER           :: header1, header2
!=============================================
INQUIRE(IOLENGTH=LREC) q
! Write binary output file
write(filename,'(A)') 'hw12_output.dat'
open(20, file=filename, status='UNKNOWN', action='write', &
     form='unformatted',access='direct',recl=LREC)

! Loop through each file
n = 1
Do i = 1, num_files
  ! Generate filename
  write(filename, '(A, I5.5, A)') 'input_', i, '.csv'
  ! Open the file
  open(10, file='./input/'//filename, status='old', &
        action='read')
  ! Read data from the file
  READ(10,*) header1
  READ(10,*) header2
  Do j = 1, num_levels
    read(10, *) lev(j),q(j),u(j),v(j)
  End do
  write(*,*)u
  write(20,rec=n)q     !write the data into new file
  n=n+1
  write(20,rec=n)u
  n=n+1
  write(20,rec=n)v
  n=n+1

  ! Close the file
  close(10)
END DO

close(20)

write(*,*) n

end program hw12


In [ ]:
DSET hw12_output.dat
UNDEF -999.0
TITLE Sample GrADS Data
XDEF 1 LINEAR 116.0 1.0
YDEF 1 LINEAR 18.0 1.0
ZDEF 37 LEVELS 1000 975 950 925 900 875 850 825 800 775 750 700 650 600 550 500 450 400 350 300 250 225 200 175 150 125 100 70 50 30 20 10 7 5 3 2 1
TDEF 200 LINEAR 00Z01JAN2001 1dy
VARS 3
q 37 99 Specific_Humidity
u 37 99 U_Component_of_Wind
v 37 99 V_Component_of_Wind
ENDVARS


In [ ]:
'reinit'

*open the file
'open hw12_output.ctl'

'set parea 1 10.5 1.8 7.5'
*set the env
'set lev 1000 300 '
'set t 1 200'

'set grads off'
'set timelab off'

*plot the qv
'set gxout shaded'
'define q1 = q*1000'
'set ylevs 1000 900 800 700 500 300'
'd q1'
'cbar'

*plot wind vector
'set gxout vector'
'd skip(u,1,5);v'
'set cthick 10'




*add title
'draw xlab Time'
'draw ylab Pressure [mb]'
'draw title Time series of qv(shaded, g/kg) and wind @(116E, 18N)'

*save the picture
'printim hw12.png white'

*close the fiile
'close 1'


In [ ]:
#hw9

PROGRAM hw9
IMPLICIT NONE
!===============
! Declare the variables
INTEGER :: i ,j,y, m, rr , divisor ,h
INTEGER,PARAMETER :: max_years = 5, max_months = 60
CHARACTER(10) :: date(max_months)
REAL :: temperature(max_months), result(max_months), ss
!REAL :: p, ave_rain,h
! ===============================================

!input the first year
WRITE(*,*)'Please input the first year (2000, 2005, 2010, 2015) :'
read(*,*) y

!get the current input
IF ((y/=2000.) .AND. (y/=2005.) .AND. (y/=2010.) .AND.(y/=2015.) ) THEN
    WRITE(*,*)'WRONG  INPUT !!'
    STOP
END IF

!input the running mean
WRITE(*,*)'Please input n for running mean :'
read(*,*) m

!get current input
h=mod(m,2)
IF ((h==0. ) .OR. (m>60.)) THEN
   WRITE(*,*)'WRONG  INPUT !!'
   STOP
END IF

!get the month need to add
rr  = (m-1)/2

!yaer 2000
IF (y==2000.) THEN
  OPEN(UNIT=10,FILE='t_2000.txt',FORM='FORMATTED',STATUS='OLD')
  OPEN(UNIT=20,FILE='running_2000.txt',FORM='FORMATTED',STATUS='UNKNOWN')
  !first loop for load data
  DO i = 1,60
     READ(10,*)date(i),temperature(i)

  END DO
  !second loop for calculate sliding avg
  DO i = 1,60
     result(i) = 0.0
     divisor = 0.0
     ss = 0.
     DO j = max(1, i - rr), min(max_months, i + rr)
        ss = ss + temperature(j)
        divisor = divisor + 1.0 ! count for whole interval
     END DO


     !input data to the new file
     IF (i - (m - 1) / 2 < 1 .OR. i + (m - 1) / 2 > max_months) THEN
        WRITE(20,*)date(i),'*'
     ELSE
        result(i) = ss/divisor  ! to get current sliding avg
        WRITE(20,*)TRIM(date(i)),result(i)
     END IF

  END DO


In [ ]:
#hw10

PROGRAM hw10
IMPLICIT NONE
! This FORTRAN program reads the monthly rainfall
! data from the CWA Taipei Station in 2001–2020
! Output formatted rainfall values, 20-year mean
! and standard deviation of each mon in tabular form
! as text file

! ===============================================
! Declare the variables
INTEGER :: i, j, k, a, m, s, t, h, l, ll, rain
REAL, DIMENSION(12) :: rain_tot
REAL, DIMENSION(69, 41) :: da,t90
REAL, DIMENSION(5) :: month
REAL, DIMENSION(2829) ::  sorted
CHARACTER(4) :: year
CHARACTER(10) :: i_str
REAL :: minimum
INTEGER :: file_unit, iostat
! ===============================================


!inpot the year
WRITE(*, *) 'Please enter the year (1960~2020) :'
READ(*, *) year

!get the current year
IF (year > '2020' .or. year < '1960') THEN
    WRITE(*, *) 'Wrong input'
    STOP
END IF

!open the new file
OPEN(20, FILE=TRIM(ADJUSTL(year)) // '.txt', FORM='formatted')
!write the frist line
WRITE(20, 100) 'month',',','min',',','max',',','mean',',','90th'
100 FORMAT(5X,A5,A1,7X,A3,A1,7X,A3,A1,6X,A4,A1,6X,A4)

!get the 1-12month data
DO i = 1, 12
    WRITE(i_str, '(I0)') i
    IF (i >= 10) THEN
        OPEN(10, FILE='./TCCIP_mmday/' // TRIM(ADJUSTL(year)) // '.'//TRIM(ADJUSTL(i_str)) // '.txt', FORM='formatted')
    ELSE
        OPEN(10, FILE='./TCCIP_mmday/' // TRIM(ADJUSTL(year)) // '.0'//TRIM(ADJUSTL(i_str))// '.txt', FORM='formatted')
    END IF

    !get the month
    a = INT(i)

    !use 2-Darray to load data
    DO m = 1, 69
        READ(10, '(41(1X,F7.2,1X))', IOSTAT=iostat) t90(m,:)
    END DO

    !turn array into 1-D to get the sequence
    DO m=1,69
      DO j = 1,41
        sorted((m - 1) * 41 + j) = t90(m, j)
      END DO
    END DO

    DO s = 1, 2828
        h = s
        minimum = sorted(s)

        DO t = s + 1, 2829
            IF (sorted(t) < minimum) THEN
                h = t
                minimum = sorted(h)
            END IF
        END DO

        sorted(h) = sorted(s)   !back to order
        sorted(s) = minimum

    END DO
    !get the min,max,avg
    month(2) = MINVAL(sorted,sorted>=0.)
    month(3) = MAXVAL(sorted)
    month(4) = NINT((SUM(sorted, sorted >= 0.) / COUNT(sorted >= 0.))*100.0) / 100.0

    !get the 90percent
    l = COUNT(sorted>=0. ) * 0.9
    ll = INT(l)

    IF (l == REAL(ll)) THEN
        month(5) = sorted(ll+count(sorted<0.))
    ELSE
        month(5) = NINT(((sorted(ll+count(sorted<0.)) + sorted(ll+1+count(sorted<0)))/ 2.0) * 100.0) /100.0
    END IF

   !write into the new file
    WRITE(20, 101) a, ',', month(2), ',', month(3),',',month(4),',',month(5)
    101 FORMAT(7X,I2,4(A1, 4X, F6.2))
    !close the file
    200 CLOSE(10)
END DO

!close the file
CLOSE(20)
END PROGRAM hw10


In [ ]:
FUNCTION SUMIVT(ivtx, ivty)
IMPLICIT NONE
REAL :: SUMIVT
REAL, INTENT(in) :: ivtx, ivty
!Caculate the sumivt
SUMIVT = SQRT(ivtx**2+ivty**2)
RETURN

END FUNCTION SUMIVT

SUBROUTINE hw11_sub(p,q,u,v,datalen,ivtx,ivty,ivt)
IMPLICIT NONE

! ===============================================
! Declare the variables
REAL, INTENT(in), DIMENSION(datalen) :: p, q, u, v
INTEGER, INTENT(in)                  :: datalen
REAL, INTENT(out)                    :: ivtx, ivty, ivt
INTEGER                              :: i
REAL, PARAMETER                      :: GR = -9.8
REAL                                 :: SUMIVT
!============================================
! Calculate the ivt with input
ivtx = 0.0
ivty = 0.0

DO i = 1,datalen-1
    ivtx = ivtx + ((q(i+1)*u(i+1)+q(i)*u(i))*(p(i+1)-p(i))*100/2/GR)
END DO

DO i = 1,datalen-1
    ivty = ivty + ((q(i+1)*v(i+1)+q(i)*v(i))*(p(i+1)-p(i))*100/2/GR)
END DO


ivt = SUMIVT(ivtx, ivty)

RETURN
END SUBROUTINE hw11_sub


In [ ]:
#hw11

PROGRAM hw11_main

!===================================
! Declare the variables
INTEGER             :: i, j, ioflag,total_arg
REAL, DIMENSION(37) :: p, q, u, v
REAL                :: ivtx,ivty,num,  ivt
CHARACTER      :: numstr*30, screen*30
CHARACTER           :: header1, header2
! ==============================================



! input argument sitting
total_arg = iargc()
CALL getarg(1, screen)
IF (total_arg == 1) THEN
  READ(screen,*) num
ELSEIF (total_arg == 0) THEN
  WRITE(*,*) 'Please give an argument (1~200) when running this',&
  'program'
  STOP
END IF



! Open the data file
IF ((NINT(num)<100) .AND. (NINT(num)>=10)) THEN
  WRITE(numstr,'(I2)') NINT(num)
  OPEN(UNIT=10,FILE='./input/input_000'//TRIM(numstr)//'.csv',&
     FORM='FORMATTED',STATUS='OLD')
ELSEIF (NINT(num)<10) THEN
  WRITE(numstr,'(I1)') NINT(num)
  OPEN(UNIT=10,FILE='./input/input_0000'//TRIM(numstr)//'.csv',&
     FORM='FORMATTED',STATUS='OLD')
ELSE
  WRITE(numstr,'(I3)') NINT(num)
  OPEN(UNIT=10,FILE='./input/input_00'//TRIM(numstr)//'.csv',&
     FORM='FORMATTED',STATUS='OLD')
ENDIF

! Read the header
READ(10,*) header1
READ(10,*) header2
! Read the vertical profile
DO i = 1,37
    READ(10,*) p(i),q(i),u(i),v(i)
END DO

! Call subroutine to calculate the ivtx,ivty and ivt
CALL hw11_sub(p,q,u,v,COUNT(p>0.),ivtx,ivty,ivt)


! Output the results to the screen
WRITE(*,'(3X,A23,F7.3,A7)') 'The value of IVTx is   ',ivtx,&
                         ' kg/m/s'
WRITE(*,'(3X,A23,F7.3,A7)') 'The value of IVTy is   ',ivty,&
                         ' kg/m/s'
WRITE(*,'(A26,F7.3,X,A6)') 'The magnitude of IVT is   ',ivt,&
                         'kg/m/s'




END PROGRAM hw11_main


In [ ]:
#hw11

#!/bin/bash

f95 hw11_main.f95 hw11_sub.f95 -o hw11.exe

./hw11.exe


In [ ]:
#hw12

PROGRAM hw12
!=====================
implicit none
character(50) :: filename
integer, parameter :: num_files = 200
integer, parameter :: num_levels = 37
real, dimension(num_levels) :: lev, q, u, v
integer :: i, j, k,  status, n, LREC
CHARACTER           :: header1, header2
!=============================================
INQUIRE(IOLENGTH=LREC) q
! Write binary output file
write(filename,'(A)') 'hw12_output.dat'
open(20, file=filename, status='UNKNOWN', action='write', &
     form='unformatted',access='direct',recl=LREC)

! Loop through each file
n = 1
Do i = 1, num_files
  ! Generate filename
  write(filename, '(A, I5.5, A)') 'input_', i, '.csv'
  ! Open the file
  open(10, file='./input/'//filename, status='old', &
        action='read')
  ! Read data from the file
  READ(10,*) header1
  READ(10,*) header2
  Do j = 1, num_levels
    read(10, *) lev(j),q(j),u(j),v(j)
  End do
  write(*,*)u
  write(20,rec=n)q     !write the data into new file
  n=n+1
  write(20,rec=n)u
  n=n+1
  write(20,rec=n)v
  n=n+1

  ! Close the file
  close(10)
END DO

close(20)

write(*,*) n

end program hw12


In [ ]:
DSET hw12_output.dat
UNDEF -999.0
TITLE Sample GrADS Data
XDEF 1 LINEAR 116.0 1.0
YDEF 1 LINEAR 18.0 1.0
ZDEF 37 LEVELS 1000 975 950 925 900 875 850 825 800 775 750 700 650 600 550 500 450 400 350 300 250 225 200 175 150 125 100 70 50 30 20 10 7 5 3 2 1
TDEF 200 LINEAR 00Z01JAN2001 1dy
VARS 3
q 37 99 Specific_Humidity
u 37 99 U_Component_of_Wind
v 37 99 V_Component_of_Wind
ENDVARS


In [ ]:
'reinit'

*open the file
'open hw12_output.ctl'

'set parea 1 10.5 1.8 7.5'
*set the env
'set lev 1000 300 '
'set t 1 200'

'set grads off'
'set timelab off'

*plot the qv
'set gxout shaded'
'define q1 = q*1000'
'set ylevs 1000 900 800 700 500 300'
'd q1'
'cbar'

*plot wind vector
'set gxout vector'
'd skip(u,1,5);v'
'set cthick 10'




*add title
'draw xlab Time'
'draw ylab Pressure [mb]'
'draw title Time series of qv(shaded, g/kg) and wind @(116E, 18N)'

*save the picture
'printim hw12.png white'

*close the fiile
'close 1'
